# 额外周末练习 — 第 2 周

## 练习目标

运用第 2 周学到的一切，为第 1 周练习中构建的**技术问答器**做一个完整原型。

应包括：

- **Gradio UI**
- **流式输出**（本实现里工具循环后给出最终回复；可按需再加 stream）
- 用 **system prompt** 增加专业性
- （加分）演示 **Tools**：写代码 / 跑代码
- （更大胆）**语音输入** + **语音回复**

这有无数商业应用：语言导师、公司入职方案、课程伴学 AI 等。

## 怎么跑

1. `.env` 配置 `OPENAI_API_KEY`
2. 从上到下运行；最后一格会打开 Gradio
3. 可文字提问，也可用麦克风；需要工具时模型会调用 `create_python_code` / `run_python_code`


## 项目计划（Plan）

分阶段实现，便于 checkpoint：

1. 添加 **Python 代码执行** 工具（`run_python_code`）
2. 添加 **Python 代码生成** 工具（`create_python_code`）
3. 给主 LLM 挂上 tools，并写好 system prompt
4. 加上 Gradio UI，以及音频回复能力
5. **Checkpoint 1** — 提交文件
6. 增加绘图（`plot.png`）与相关展示逻辑
7. **Checkpoint 2**
8. 加入语音输入（Whisper / transcription）
9. 提交作品


In [ ]:
# ========== 导入：环境、OpenAI、Gradio ==========

# 标准库 os：读环境变量、检查文件是否存在（如 plot.png）
import os
# 标准库 json：把工具执行结果序列化成字符串塞进 tool 消息
import json
# load_dotenv：从 .env 加载密钥
from dotenv import load_dotenv
# OpenAI 客户端：Chat Completions + 音频转写/TTS
from openai import OpenAI
# Gradio：Blocks / Chatbot / Audio 搭界面
import gradio as gr


In [ ]:
# ========== 初始化：密钥检查、模型常量、客户端、DB 文件名 ==========

# override=True：.env 覆盖进程里已有同名变量
load_dotenv(override=True)

# 读取 OPENAI_API_KEY
openai_api_key = os.getenv('OPENAI_API_KEY')
if openai_api_key:
    # 只打印前缀，避免泄露完整密钥
    print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("OpenAI API Key not set")

# 主对话模型 id（保持原字符串，不改）
MODEL = "gpt-4.1-mini"
# 创建默认 OpenAI 客户端
openai = OpenAI()

# 预留的 SQLite 文件名常量（本练习主线是代码工具，不一定用到）
DB = "prices.db"


In [ ]:
# ========== 系统提示词：规定两个工具的严格使用规则 ==========
# 整段英文是发给模型的 prompt，必须原样保留（含字符串内的标记）

system_message = """You are a technical AI assistant with access to two tools:

1. create_python_code → generates Python code
2. run_python_code → executes Python code

Your goal is to answer user queries accurately using reasoning and tools when appropriate.

---

# 【注】## TOOL USAGE RULES (STRICT)

You must follow these rules exactly:

1. If the user asks for Python code explicitly:
   → Call create_python_code ONLY
   → Do NOT call run_python_code

2. If the user provides Python code and asks for its output:
   → Call run_python_code ONLY
   → Pass the given code directly without modification

3. If the user asks a problem that can benefit from computation 
   (math, logic, data processing, plotting, etc.):
   → FIRST call create_python_code
   → THEN pass the EXACT SAME code to run_python_code
   → DO NOT modify, edit, or reinterpret the code in any way

4. NEVER write or modify code yourself if create_python_code is used
   → The code from create_python_code must be passed AS-IS to run_python_code

---

# 【注】## IMPORTANT CONSTRAINTS

- You are NOT allowed to alter tool outputs
- You must NOT regenerate or rewrite code after it is created
- You must NOT skip steps in the tool chain when both tools are needed
- Always rely on tools for computation instead of doing it manually
- Always save the plot as "plot.png" in the current directory.
- Do NOT use plt.show().

---

# 【注】## RESPONSE BEHAVIOR

- Use tools whenever they improve accuracy
- Keep final answers clear and concise
- Do not mention internal tool logic unless necessary
- Assume plots or files generated will be shown to the user automatically
- Do NOT say "I cannot display images"

---

# 【注】## THINKING

- Think step by step before deciding which tool(s) to use
- Choose the minimal correct sequence of actions

---

Your objective is to correctly decide:
- whether to use tools
- which tool(s) to use
- and in what order

while strictly following all rules above."""


In [ ]:
# ========== 工具实现：在子进程里安全地跑一段 Python 代码 ==========

# subprocess：启动子进程执行代码文件
import subprocess
# tempfile：生成临时 .py 文件
import tempfile
# os：删除临时文件、判断路径
import os
# sys：拿到当前解释器路径 sys.executable，保证用同一 Python
import sys

def run_python_code(code: str):
    try:
        # 创建临时 .py 文件，写入模型（或测试）给出的代码；delete=False 以便稍后 subprocess 读取
        with tempfile.NamedTemporaryFile(delete=False, suffix=".py", mode="w") as temp_file:
            temp_file.write(code)
            temp_path = temp_file.name

        # 用当前解释器运行该文件；capture_output 收集 stdout/stderr；timeout 防死循环
        result = subprocess.run(
            [sys.executable, temp_path],
            capture_output=True,
            text=True,
            timeout=30  # prevent infinite loops
        )

        # 结构化返回，方便后面 json.dumps 塞进 tool 消息
        return {
            "stdout": result.stdout,
            "stderr": result.stderr,
            "returncode": result.returncode
        }

    except subprocess.TimeoutExpired:
        # 超时也返回统一结构，避免上层崩溃
        return {
            "stdout": "",
            "stderr": "Execution timed out",
            "returncode": -1
        }

    finally:
        # 无论成功失败都尽量删掉临时文件
        if os.path.exists(temp_path):
            os.remove(temp_path)


In [ ]:
# ========== 本地冒烟测试：不经过 LLM，直接跑一小段代码 ==========

# 写一段简单可执行代码字符串（影响行为的英文 print 内容保持原样）
code = """
print("Hello World")
x = 5
print(x * 2)
"""

# 调用上面的执行工具，应看到 stdout 里有 Hello World 与 10
run_python_code(code)


In [ ]:
# ========== 第二个工具：让「写代码专用」LLM 生成可执行 Python ==========

def create_python_code(question: str) -> str:
    # 本函数内部也可单独指定模型（与外层 MODEL 同值，保持原样）
    MODEL = "gpt-4.1-mini"
    # 代码生成器的 system prompt：要求只输出可执行代码（字符串内容不翻译）
    system_message = """You are a Python code generator.

Your task is to write complete, executable Python code that solves the user's query.

STRICT RULES:
- Output ONLY valid Python code. Do NOT include explanations.
- Do NOT use markdown formatting (no ``` or ```python).
- The code must run without errors as-is.
- Include all necessary imports.
- Do not assume any pre-defined variables.
- Always use print() to display final results.
- Always use UTF-8 compatible characters
- Avoid special unicode symbols like π, use 'pi' instead
- Save plots as "plot.png" in the current directory

FOR PLOTS:
- If the task involves plotting, use matplotlib.
- Always save the plot as "plot.png" in the current directory.
- Do NOT use plt.show().
- After saving, print exactly: plot saved at plot.png and shown to the user.

CODE QUALITY:
- Keep code clean and readable.
- Avoid unnecessary complexity.
- Ensure correctness over cleverness.

Your output must be directly executable Python code and nothing else."""

    # 单轮：system 定规则 + user 放问题
    messages = [{"role": "system", "content": system_message}, {"role": "user", "content": question}]
    # 非流式一次拿完整代码字符串
    response = openai.chat.completions.create(model=MODEL, messages=messages)
    return response.choices[0].message.content


In [ ]:
# 单独测试代码生成工具：问「前 10 个自然数之和」
(create_python_code("What is the sum of the first 10 natural numbers?"))


In [ ]:
# 串联测试：先生成代码，再立刻用 run_python_code 执行
run_python_code(create_python_code("What is the sum of the first 10 natural numbers?"))


In [ ]:
# ========== 把两个 Python 函数描述成 OpenAI tools schema ==========

# 执行代码工具的描述（name 必须与函数名 / handle 分支一致）
run_code_function = {
    "name": "run_python_code",
    "description": "Run python code and get the output. The input should be a string of python code. The output will be a dictionary with keys stdout, stderr and returncode.",
    "parameters": {
        "type": "object",
        "properties": {
            "code": {
                "type": "string",
                "description": "The python code to run"
            }
        },
        "required": ["code"]
    }
}

# 生成代码工具的描述
create_code_function = {
    "name": "create_python_code",
    "description": "Create python code to answer a question or create a plot. The input should be a string of the question. The output will be a string of python code that can be run to get the answer to the question.",
    "parameters": {
        "type": "object",
        "properties": {
            "question": {
                "type": "string",
                "description": "The question to answer with python code"
            }
        },
        "required": ["question"]
    }
}


In [ ]:
# ========== 组装 tools 列表，供 chat.completions.create(tools=...) 使用 ==========

tools = [{"type": "function", "function": run_code_function}, {"type": "function", "function": create_code_function}]


In [ ]:
# ========== 语音转文字：OpenAI transcriptions API ==========

def transcribe_audio(audio_path):
    # 以二进制打开 Gradio 录下的音频文件路径
    with open(audio_path, "rb") as audio_file:
        # 调用转写模型；model id 保持原样
        transcript = openai.audio.transcriptions.create(
            model="gpt-4o-mini-transcribe",
            file=audio_file
        )
    # 返回纯文本，后续当作 user 消息
    return transcript.text


In [ ]:
# ========== Gradio 回调：麦克风音频 → 转写 → 追加到聊天历史 ==========

def handle_voice_input(audio, history):
    # 没录到声音就原样返回历史
    if audio is None:
        return history

    # 转写为文本
    text = transcribe_audio(audio)
    print(f"Transcribed: {text}")

    # 追加一条 user 消息（OpenAI messages 风格）
    return history + [{"role": "user", "content": text}]


In [ ]:
# ========== TTS：把助手文本转成语音字节（OpenAI speech API） ==========

def talker(message):
    response = openai.audio.speech.create(
      model="gpt-4o-mini-tts",
      voice="coral",    # Also, try replacing onyx with alloy or coral
      input=message
    )
    # 返回音频二进制内容，供 Gradio Audio 播放
    return response.content

# 启动时先合成一句问候，后面 chat 暂时固定返回这个 voice（见下方注释逻辑）
voice = talker("Hello, how can I help you today?")


In [ ]:
# ========== 主对话循环：清洗历史 → tools while → 可选插入 plot 图片 ==========

def chat(history):
    # 标记本轮工具是否生成了 plot.png
    image_generated = False
    # 发给模型的「干净」历史（去掉图片元组等 Gradio 特殊 content）
    clean_history = []

    for h in history:
        content = h["content"]

        # 若 content 是元组（常表示图片路径），跳过，避免污染 LLM messages
        if isinstance(content, tuple):
            continue

        # 若 content 是列表，只拼接其中的字符串片段
        if isinstance(content, list):
            content = " ".join([c for c in content if isinstance(c, str)])

        clean_history.append({
            "role": h["role"],
            "content": content
        })

    # system + 清洗后的多轮对话
    messages = [{"role": "system", "content": system_message}] + clean_history

    # 第一次调用：带 tools，可能直接答或请求工具
    response = openai.chat.completions.create(model=MODEL, messages=messages, tools=tools)
    print(f"Initial response: {response.choices[0].message.content}, finish_reason: {response.choices[0].finish_reason}")

    # 多轮工具：执行 → 追加 → 再问（继续传 tools）
    while response.choices[0].finish_reason=="tool_calls":
        message = response.choices[0].message
        print(f"Tool call message: {message}")
        responses, image_generated = handle_tool_calls(message)
        print(f'tool call responses: {responses}')
        messages.append(message)
        messages.extend(responses)
        response = openai.chat.completions.create(model=MODEL, messages=messages, tools=tools)
        print(f"Response after tool calls: {response}")

    # 最终文本回复追加到 Gradio history
    reply = response.choices[0].message.content
    history += [{"role": "assistant", "content": reply}]

    # 若工具跑出了图，再追加一条「图片消息」（tuple 路径约定）
    image_path = "plot.png"
    if image_generated and os.path.exists(image_path):
        history += [{
            "role": "assistant",
            "content": (image_path,)
        }]
        print(f'Image generated at {image_path}')
        # 若需要用完即删，可取消下一行注释：os.remove(image_path)

    # 若要对「本轮 reply」实时 TTS，可改为：voice = talker(reply)
    # 当前逻辑保持原样：返回单元格开头预生成的 voice

    return history, voice


In [ ]:
# ========== 工具分发器：按 function.name 执行并组装 tool 消息 ==========

def handle_tool_calls(message):
    responses = []
    image_generated = False

    for tool_call in message.tool_calls:
        # 分支：执行代码
        if tool_call.function.name == "run_python_code":
            arguments = json.loads(tool_call.function.arguments)
            code = arguments.get('code')
            output = run_python_code(code)
            responses.append({
                "role": "tool",
                # 字典 → JSON 字符串，模型才能读 stdout/stderr
                "content": json.dumps(output),
                "tool_call_id": tool_call.id
            })
            # 约定：若 stdout 提到 plot.png，则 UI 稍后插入图片
            if "plot.png" in output.get("stdout", ""):
                image_generated = True

        # 分支：生成代码（只返回代码字符串给模型）
        elif tool_call.function.name == "create_python_code":
            arguments = json.loads(tool_call.function.arguments)
            question = arguments.get('question')
            code = create_python_code(question)
            responses.append({
                "role": "tool",
                "content": code,
                "tool_call_id": tool_call.id
            })

    # 同时返回工具消息列表 + 是否出图标志
    return responses, image_generated


In [ ]:
# ========== Gradio UI：文本提交 / 麦克风 → 共用 chat ==========

# 把用户输入框内容放入 chatbot history，并清空输入框
def put_message_in_chatbot(message, history):
        print(f"User message: {message}")
        return "", history + [{"role":"user", "content":message}]

# UI 布局：聊天 + 音频输出 + 文本框 + 麦克风
with gr.Blocks() as ui:
    with gr.Row():
        chatbot = gr.Chatbot(height=400, type="messages")

    with gr.Row():
        # autoplay=True：收到音频就自动播放
        audio_output = gr.Audio(autoplay=True)

    with gr.Row():
        message = gr.Textbox(label="Chat with our AI Assistant:")

    with gr.Row():
        # 麦克风录音，type=filepath 方便交给 transcription API
        audio_input = gr.Audio(sources=["microphone"], type="filepath")

    # 文本：先入历史，再跑 chat（更新 chatbot + 音频）
    message.submit(
        put_message_in_chatbot,
        inputs=[message, chatbot],
        outputs=[message, chatbot]
    ).then(
        chat,
        inputs=chatbot,
        outputs=[chatbot, audio_output]
    )

    # 语音：转写入历史，再跑 chat
    audio_input.change(
        handle_voice_input,
        inputs=[audio_input, chatbot],
        outputs=chatbot
    ).then(
        chat,
        inputs=chatbot,
        outputs=[chatbot, audio_output]
    )
# inbrowser=True：启动后尝试自动打开浏览器
ui.launch(inbrowser=True)
